### 1. 주차장 데이터(parking_preprocessed.csv)와 매칭하여 전기차 충전기 보급률이나 현황을 분석

목적: 기존데이터의 공영주차장에 실제로 전기차충전기를 얼마나 가지고 있는 지 실제보급률을 확인 (but 일렉베리 사이트에서 추출한 전기차충전소 데이터가 모든 전기차충전소를 포함하고 있지는 않음)

#### 1.1 전기차충전소 지도 사이트 - 일렉베리 : 웹크롤링

목적: 서울, 공영주차장을 필터링하여 공영주차장 내에 전기차충전기가 있는 주차장 수집

In [2]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

# 1. 브라우저 옵션 설정
chrome_options = Options()
chrome_options.add_experimental_option("detach", True)
# 불필요한 로그를 줄여주는 설정
chrome_options.add_argument("--log-level=3")

# 2. 드라이버 실행 (캐시 삭제 후 실행하면 자동으로 새로 받습니다)
try:
    # Service나 ChromeDriverManager 없이 아래처럼만 실행해 보세요.
    driver = webdriver.Chrome(options=chrome_options)
    print("드라이버가 성공적으로 실행되었습니다!")
except Exception as e:
    print(f"드라이버 실행 실패: {e}")

# 3. 크롤링 함수 (구조만 확인용)
def check_page():
    try:
        url = "https://www.elecvery.com/ko/map?placeTypeList=PUB_PARKING"
        driver.get(url)
        time.sleep(3)
        print("페이지 접속 성공!")
    finally:
        # driver.quit() # 확인을 위해 열어둠
        pass

if __name__ == "__main__":
    check_page()

드라이버가 성공적으로 실행되었습니다!
페이지 접속 성공!


목표: 서울 지역 공영주차장의 이름, 주소, 충전기 수 수집

환경: VS Code, Python, Selenium (정상 작동 확인됨)

추가 필요 사항: 리스트 스크롤링 및 정확한 데이터 추출 로직

In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

chrome_options = Options()
chrome_options.add_experimental_option("detach", True)
driver = webdriver.Chrome(options=chrome_options)

def crawl_seoul_parking_limit_15():
    try:
        url = "https://www.elecvery.com/ko/map?placeTypeList=PUB_PARKING"
        driver.get(url)
        time.sleep(5)

        # '지역 충전소' 탭 클릭
        try:
            tab = driver.find_element(By.XPATH, "//*[contains(text(), '지역 충전소')]")
            driver.execute_script("arguments[0].click();", tab)
            time.sleep(3)
        except: pass

        all_results = []
        current_page = 1
        
        while current_page <= 15: # 15페이지까지만 작동
            print(f"\n[현재 {current_page}페이지 수집 중]")
            time.sleep(4) # 데이터 로딩을 위해 충분히 대기

            # 리스트 데이터 수집
            items = driver.find_elements(By.TAG_NAME, "li")
            page_data_count = 0
            for item in items:
                text = item.text
                if "서울" in text:
                    lines = [l.strip() for l in text.split('\n') if l.strip()]
                    if len(lines) >= 2:
                        data = {"이름": lines[0], "주소": lines[1]}
                        if data not in all_results:
                            all_results.append(data)
                            page_data_count += 1
            
            print(f"-> {page_data_count}개 수집 완료 (누적: {len(all_results)}건)")

            # 15페이지 수집이 끝났으면 바로 종료
            if current_page == 15:
                print("\n목표한 15페이지 수집이 완료되었습니다. 종료합니다.")
                break

            # 다음 페이지 이동 로직
            try:
                next_num = current_page + 1
                next_btns = driver.find_elements(By.XPATH, f"//*[text()='{next_num}']")
                
                if next_btns:
                    driver.execute_script("arguments[0].click();", next_btns[0])
                    print(f"-> {next_num}번 숫자 클릭 성공")
                else:
                    # 숫자가 없으면 화살표 클릭 (직접 주신 XPath 활용)
                    print(f"-> {next_num}번 숫자가 없어 화살표(>)를 클릭합니다.")
                    exact_arrow_xpath = "/html/body/div[1]/div/div[2]/aside/div[2]/div/div[4]/div[3]/div/button[7]"
                    arrow_btn = driver.find_element(By.XPATH, exact_arrow_xpath)
                    driver.execute_script("arguments[0].click();", arrow_btn)
                
                current_page += 1
                time.sleep(2)
            except Exception as e:
                print(f"이동 중 오류 발생 혹은 끝 도달: {e}")
                break

        # 결과 저장
        df = pd.DataFrame(all_results).drop_duplicates()
        df.to_csv(r"C:\\Users\\seonu\\Documents\\axis3-self-consumption\\canopy\\output\\results\\seoul_parking_simple_list.csv", index=False, encoding="utf-8-sig")
        print(f"\n[최종 수집 완료] 총 {len(df)}건 저장되었습니다.")

    except Exception as e:
        print(f"오류: {e}")

if __name__ == "__main__":
    crawl_seoul_parking_limit_15()


[현재 1페이지 수집 중]
-> 10개 수집 완료 (누적: 10건)
-> 2번 숫자 클릭 성공

[현재 2페이지 수집 중]
-> 10개 수집 완료 (누적: 20건)
-> 3번 숫자 클릭 성공

[현재 3페이지 수집 중]
-> 10개 수집 완료 (누적: 30건)
-> 4번 숫자 클릭 성공

[현재 4페이지 수집 중]
-> 10개 수집 완료 (누적: 40건)
-> 5번 숫자 클릭 성공

[현재 5페이지 수집 중]
-> 10개 수집 완료 (누적: 50건)
-> 6번 숫자가 없어 화살표(>)를 클릭합니다.

[현재 6페이지 수집 중]
-> 10개 수집 완료 (누적: 60건)
-> 7번 숫자 클릭 성공

[현재 7페이지 수집 중]
-> 10개 수집 완료 (누적: 70건)
-> 8번 숫자 클릭 성공

[현재 8페이지 수집 중]
-> 10개 수집 완료 (누적: 80건)
-> 9번 숫자 클릭 성공

[현재 9페이지 수집 중]
-> 6개 수집 완료 (누적: 86건)
-> 10번 숫자 클릭 성공

[현재 10페이지 수집 중]
-> 9개 수집 완료 (누적: 95건)
-> 11번 숫자가 없어 화살표(>)를 클릭합니다.

[현재 11페이지 수집 중]
-> 10개 수집 완료 (누적: 105건)
-> 12번 숫자 클릭 성공

[현재 12페이지 수집 중]
-> 9개 수집 완료 (누적: 114건)
-> 13번 숫자 클릭 성공

[현재 13페이지 수집 중]
-> 10개 수집 완료 (누적: 124건)
-> 14번 숫자 클릭 성공

[현재 14페이지 수집 중]
-> 10개 수집 완료 (누적: 134건)
-> 15번 숫자 클릭 성공

[현재 15페이지 수집 중]
-> 10개 수집 완료 (누적: 144건)

목표한 15페이지 수집이 완료되었습니다. 종료합니다.

[최종 수집 완료] 총 144건 저장되었습니다.


#### 1.2 주차장 파일과 일렉베리 매칭

**매칭 방식:**
1. 두 파일을 읽어옵니다.
2. '주차장명' 혹은 '주소'를 기준으로 Left Join을 수행합니다. (기존 데이터 기준)
3. 이름이 정확히 일치하지 않을 경우를 대비해, 이름에 특정 키워드가 포함되어 있는지 확인하는 방식으로 매칭률을 높일 수 있습니다.

In [28]:
import pandas as pd

# 1. 파일 경로 설정
existing_data_path = r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\data\processed\parking_preprocessed.csv'
crawled_data_path = r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\output\results\seoul_parking_simple_list.csv'

try:
    # 2. 데이터 불러오기
    # 기존 데이터 (pklt_nm, addr 포함)
    df_existing = pd.read_csv(existing_data_path)
    # 크롤링 데이터 (이름, 주소 포함)
    df_crawled = pd.read_csv(crawled_data_path)

    # 3. 매칭률을 높이기 위한 전처리 (공백 제거)
    # 기존 데이터의 'pklt_nm'과 크롤링 데이터의 '이름'을 비교 키로 사용
    df_existing['match_key'] = df_existing['pklt_nm'].str.replace(r'\s+', '', regex=True)
    df_crawled['match_key'] = df_crawled['이름'].str.replace(r'\s+', '', regex=True)

    # 4. 데이터 매칭 (Left Join)
    # 기존 주차장 리스트를 유지하면서 크롤링된 정보가 있으면 옆에 붙임
    df_merged = pd.merge(
        df_existing, 
        df_crawled[['match_key', '이름']], 
        on='match_key', 
        how='left'
    )

    # 5. 충전소 보유 여부 생성 (크롤링 결과 '이름'이 존재하면 보유한 것)
    df_merged['ev_charger_yn'] = df_merged['이름'].apply(lambda x: 'Y' if pd.notnull(x) else 'N')

    # 6. 결과 리포트 및 저장
    total_parking = len(df_existing)
    ev_parking = df_merged[df_merged['ev_charger_yn'] == 'Y'].shape[0]
    match_rate = (ev_parking / total_parking) * 100

    print(f"--- 매칭 분석 결과 ---")
    print(f"전체 공영주차장 수: {total_parking}개")
    print(f"전기차 충전소 확인됨: {ev_parking}개")
    print(f"충전기 보급률(매칭 기준): {match_rate:.2f}%")

    # 불필요해진 임시 컬럼 삭제 후 저장
    output_path = r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\output\results\parking_ev_matching_result.csv'
    df_merged.drop(columns=['match_key', '이름']).to_csv(output_path, index=False, encoding='utf-8-sig')
    
    print(f"\n[성공] 분석 결과가 다음 경로에 저장되었습니다:\n{output_path}")

except FileNotFoundError:
    print("오류: 파일을 찾을 수 없습니다. 경로가 정확한지 다시 확인해 주세요.")
except Exception as e:
    print(f"오류 발생: {e}")

--- 매칭 분석 결과 ---
전체 공영주차장 수: 742개
전기차 충전소 확인됨: 34개
충전기 보급률(매칭 기준): 4.58%

[성공] 분석 결과가 다음 경로에 저장되었습니다:
C:\Users\seonu\Documents\axis3-self-consumption\canopy\output\results\parking_ev_matching_result.csv


### 1.3 주차장 데이터(parking_preprocessed.csv)와 공공데이터(서울_EV충전소.csv)를 매칭하여 전기차 충전기 보급률이나 현황을 분석

목적: 기존데이터의 공영주차장에 실제로 전기차충전기를 얼마나 가지고 있는 지 실제보급률을 확인 (일렉베리 데이터가 누락되었을 가능성까지 고려해서 공공데이터(서울_EV충전소.csv)로 상호검증)


아직안함

---

### 2. 50m프록시 검증 - 일렉베리

프록시 기반 추정치(parking_axis3_scored.csv)와 실제 웹사이트 리스트(크롤링)를 비교하여 분석 모델이 얼마나 정확했는지 검증(Validation)

**"추정치(Proxy) vs 실측치(Crawling)"를 비교**

목적: 축3점수 산출에 있어 공공데이터의 한계를 극복하기 위해 좌표 기반 50m 프록시(Proxy)라는 공학적인 접근 방식을 사용함, 이를 실제 웹사이트 데이터로 검증(but일렉베리 사이트에서 추출한 전기차충전소 데이터가 모든 전기차충전소를 포함하고 있지는 않음)

1. 검증 전략
- 기준 데이터: API 좌표 분석이 완료된 parking_axis3_scored.csv

- 비교 데이터: seoul_parking_simple_list.csv (실제 존재 리스트)

- 검증 지표:

    1. 정합성(Accuracy): API 분석에서 충전기가 있다고 판단했는데, 실제로도 크롤링 데이터에 존재하는가?

    2. 과잉 추정(False Positive): API는 있다고 했는데, 실제 리스트에는 없는가?

    3. 과소 추정(False Negative): API는 없다고 했는데, 실제 리스트에는 있는가?

#### 2.1 텍스트 매칭(주차장명)

In [29]:
import pandas as pd

# 1. 파일 경로 설정
scored_data_path = r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\data\processed\parking_axis3_scored.csv'
crawled_data_path = r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\output\results\seoul_parking_simple_list.csv'

try:
    # 2. 데이터 불러오기
    df_scored = pd.read_csv(scored_data_path)
    df_crawled = pd.read_csv(crawled_data_path)

    # 3. 매칭을 위한 전처리 (이름 공백 제거)
    df_scored['match_key'] = df_scored['pklt_nm'].str.replace(r'\s+', '', regex=True)
    df_crawled['match_key'] = df_crawled['이름'].str.replace(r'\s+', '', regex=True)

    # 4. 데이터 병합 (Scored 데이터 기준)
    df_val = pd.merge(
        df_scored, 
        df_crawled[['match_key', '이름']], 
        on='match_key', 
        how='left'
    )

    # 5. 검증 라벨링
    # - 실제 존재 여부 (크롤링 결과 기준)
    df_val['real_exists'] = df_val['이름'].apply(lambda x: True if pd.notnull(x) else False)
    # - API 분석 결과 존재 여부 (설치수 > 0 기준)
    df_val['api_predicted'] = df_val['ev_충전기_설치수'].apply(lambda x: True if x > 0 else False)

    # 6. 오차 행렬(Confusion Matrix) 분석
    # True Positive: 둘 다 있음
    tp = len(df_val[(df_val['real_exists'] == True) & (df_val['api_predicted'] == True)])
    # False Positive: API는 있다는데 실제론 없음 (과잉)
    fp = len(df_val[(df_val['real_exists'] == False) & (df_val['api_predicted'] == True)])
    # False Negative: API는 없다는데 실제론 있음 (누락)
    fn = len(df_val[(df_val['real_exists'] == True) & (df_val['api_predicted'] == False)])
    # True Negative: 둘 다 없음
    tn = len(df_val[(df_val['real_exists'] == False) & (df_val['api_predicted'] == False)])

    print(f"--- 분석 모델 검증 결과 ---")
    print(f"1. 정답 (실제와 일치): {tp}건")
    print(f"2. 과잉 추정 (API오류/반경오류): {fp}건")
    print(f"3. 누락 (매칭실패/추정실패): {fn}건")
    print(f"4. 정확도(Precision/Recall 기반): {tp/(tp+fp):.2f} (정밀도)")

    # 7. 결과 저장
    output_path = r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\output\results\final_model_validation.csv'
    df_val.drop(columns=['match_key', '이름']).to_csv(output_path, index=False, encoding='utf-8-sig')
    
    print(f"\n[성공] 검증 데이터가 저장되었습니다: {output_path}")

except Exception as e:
    print(f"오류 발생: {e}")

--- 분석 모델 검증 결과 ---
1. 정답 (실제와 일치): 6건
2. 과잉 추정 (API오류/반경오류): 110건
3. 누락 (매칭실패/추정실패): 28건
4. 정확도(Precision/Recall 기반): 0.05 (정밀도)

[성공] 검증 데이터가 저장되었습니다: C:\Users\seonu\Documents\axis3-self-consumption\canopy\output\results\final_model_validation.csv


- 일렉베리 데이터의 한계: 일렉베리는 민간 서비스라 모든 충전소를 포함하지 않을 수 있습니다.

#### 2.2 좌표(위경도)매칭

목적: 단순 텍스트 매칭이 아닌 공간 분석(Spatial Join)을 통해 모델의 타당성을 검증했다

1. 지오코딩 및 좌표 매칭 전략
    1. 지오코딩: 일렉베리 데이터(seoul_parking_simple_list.csv)의 주소를 좌표(lat, lng)로 변환합니다. (무료 API인 Kakao나 Vworld를 추천합니다.)

    2. 좌표 병합: 변환된 좌표와 사용자님의 모델 데이터(parking_axis3_scored.csv)를 거리 기준으로 매칭합니다.

    3. 검증: 두 지점 사이의 거리가 매우 가깝다면(예: 20~30m), 실제 동일 주차장으로 간주하고 분석 모델의 정확도를 평가합니다.
2. 추천 도구: Vworld API (국가공공데이터)

In [ ]:
import requests
import pandas as pd
import time

# Vworld API 설정 (키를 발급받아 입력하세요)
API_KEY = '1642B896-879F-33A4-A900-E875136DA7A5'

# 지오코딩 함수
def get_coords_vworld(address):
    url = f"https://api.vworld.kr/req/address?service=address&request=getcoord&version=2.0&crs=epsg:4326&address={address}&refine=true&simple=false&format=json&type=road&key={API_KEY}"
    try:
        response = requests.get(url)
        data = response.json()
        if data['response']['status'] == 'OK':
            lon = data['response']['result']['point']['x']
            lat = data['response']['result']['point']['y']
            return float(lat), float(lon)
    except:
        return None, None
    return None, None

# 1. 일렉베리 데이터 로드 및 좌표 추출
df_crawled = pd.read_csv(r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\output\results\seoul_parking_simple_list.csv')

print("지오코딩 시작...")
coords = []
for addr in df_crawled['주소']:
    lat, lon = get_coords_vworld(addr)
    coords.append({'lat_c': lat, 'lon_c': lon})
    time.sleep(0.1) # API 부하 방지

df_crawled_coords = pd.concat([df_crawled, pd.DataFrame(coords)], axis=1)

# 2. 이후 단계: haversine 함수를 이용해 scored_data와 거리 매칭
# (이전에 썼던 좌표 매칭 코드를 이 데이터에 적용하면 됩니다!)

지오코딩 시작...


In [35]:
import requests
import pandas as pd
import numpy as np
import time

# ==========================================
# 1. 설정 및 경로 (본인 환경에 맞게 수정)
# ==========================================
VWORLD_API_KEY = '1642B896-879F-33A4-A900-E875136DA7A5'

crawled_input_path = r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\output\results\seoul_parking_simple_list.csv'
scored_input_path = r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\data\processed\parking_axis3_scored.csv'
geocoded_output_path = r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\output\results\seoul_parking_geocoded.csv'
final_val_path = r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\output\results\final_spatial_validation.csv'

# 하버사인 거리 계산 함수
def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371000 
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

# Vworld 지오코딩 함수
def get_coords(address):
    url = f"https://api.vworld.kr/req/address?service=address&request=getcoord&version=2.0&crs=epsg:4326&address={address}&refine=true&simple=false&format=json&type=road&key={VWORLD_API_KEY}"
    try:
        res = requests.get(url)
        json_data = res.json()
        if json_data['response']['status'] == 'OK':
            x = json_data['response']['result']['point']['x'] # 경도
            y = json_data['response']['result']['point']['y'] # 위도
            return float(y), float(x)
    except:
        return None, None
    return None, None

# ==========================================
# 2. 실행 프로세스
# ==========================================
try:
    # A. 크롤링 데이터 지오코딩
    print("Step 1: 일렉베리 데이터 지오코딩 시작...")
    df_crawled = pd.read_csv(crawled_input_path)
    
    lats, lons = [], []
    for addr in df_crawled['주소']:
        lat, lon = get_coords(addr)
        lats.append(lat)
        lons.append(lon)
        time.sleep(0.1) # API 매너 타임
        if len(lats) % 20 == 0: print(f"  - {len(lats)}개 변환 완료...")

    df_crawled['lat_c'] = lats
    df_crawled['lon_c'] = lons
    df_crawled.to_csv(geocoded_output_path, index=False, encoding='utf-8-sig')
    print(f"-> 지오코딩 완료 및 저장: {geocoded_output_path}")

    # B. 좌표 기반 거리 매칭 검증
    print("\nStep 2: 좌표 기반 거리 매칭 검증 시작...")
    df_scored = pd.read_csv(scored_input_path)
    df_geocoded = df_crawled.dropna(subset=['lat_c', 'lon_c'])

    min_dists = []
    for _, row in df_scored.iterrows():
        dists = haversine_vectorized(row['lat'], row['lot'], df_geocoded['lat_c'], df_geocoded['lon_c'])
        min_dists.append(np.min(dists))

    df_scored['dist_to_real'] = min_dists
    
    # 50m 기준 검증
    threshold = 50
    df_scored['is_real_exists'] = df_scored['dist_to_real'] <= threshold
    df_scored['model_predicted'] = df_scored['ev_충전기_설치수'] > 0

    # 지표 계산
    tp = len(df_scored[(df_scored['model_predicted'] == True) & (df_scored['is_real_exists'] == True)])
    fp = len(df_scored[(df_scored['model_predicted'] == True) & (df_scored['is_real_exists'] == False)])
    fn = len(df_scored[(df_scored['is_real_exists'] == True) & (df_scored['model_predicted'] == False)])
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    
    print(f"\n[최종 검증 결과]")
    print(f"- 일치(TP): {tp}건")
    print(f"- 과잉(FP): {fp}건")
    print(f"- 누락(FN): {fn}건")
    print(f"- 정밀도(Precision): {precision:.2f}")

    df_scored.to_csv(final_val_path, index=False, encoding='utf-8-sig')
    print(f"\n전체 분석 완료! 결과 파일: {final_val_path}")

except Exception as e:
    print(f"\n오류 발생: {e}")

Step 1: 일렉베리 데이터 지오코딩 시작...
  - 20개 변환 완료...
  - 40개 변환 완료...
  - 60개 변환 완료...
  - 80개 변환 완료...
  - 100개 변환 완료...
  - 120개 변환 완료...
  - 140개 변환 완료...
-> 지오코딩 완료 및 저장: C:\Users\seonu\Documents\axis3-self-consumption\canopy\output\results\seoul_parking_geocoded.csv

Step 2: 좌표 기반 거리 매칭 검증 시작...

[최종 검증 결과]
- 일치(TP): 3건
- 과잉(FP): 113건
- 누락(FN): 0건
- 정밀도(Precision): 0.03

전체 분석 완료! 결과 파일: C:\Users\seonu\Documents\axis3-self-consumption\canopy\output\results\final_spatial_validation.csv


**원인분석**

- 반경 50m의 밀도 문제: 서울 도심에서 50m 반경은 생각보다 굉장히 넓습니다. 주차장 좌표를 중심으로 50m 원을 그리면 그 안에 상가, 아파트, 오피스텔 충전기가 수두룩하게 들어옵니다. API 데이터는 이를 주차장 내부로 오인한 것이죠.

- 일렉베리 데이터의 특성: 오늘 크롤링한 데이터는 '지역 충전소' 리스트입니다. 일렉베리 사이트가 모든 민간/공공 충전소를 다 보여주는 것이 아니라, 특정 네트워크(제휴 등) 중심의 데이터를 보여줄 경우, 실제로는 충전소가 있는데 일렉베리 리스트에만 없어서 '과잉(FP)'으로 처리되었을 가능성이 큽니다.

- 좌표의 불일치: 주차장의 중심점 좌표와 실제 충전기가 설치된 구석진 위치 사이의 거리가 생각보다 멀 수 있습니다.

In [36]:
# ==========================================
# 2. 실행 프로세스
# ==========================================
try:
    # A. 크롤링 데이터 지오코딩
    print("Step 1: 일렉베리 데이터 지오코딩 시작...")
    df_crawled = pd.read_csv(crawled_input_path)
    
    lats, lons = [], []
    for addr in df_crawled['주소']:
        lat, lon = get_coords(addr)
        lats.append(lat)
        lons.append(lon)
        time.sleep(0.1) # API 매너 타임
        if len(lats) % 20 == 0: print(f"  - {len(lats)}개 변환 완료...")

    df_crawled['lat_c'] = lats
    df_crawled['lon_c'] = lons
    df_crawled.to_csv(geocoded_output_path, index=False, encoding='utf-8-sig')
    print(f"-> 지오코딩 완료 및 저장: {geocoded_output_path}")

    # B. 좌표 기반 거리 매칭 검증
    print("\nStep 2: 좌표 기반 거리 매칭 검증 시작...")
    df_scored = pd.read_csv(scored_input_path)
    df_geocoded = df_crawled.dropna(subset=['lat_c', 'lon_c'])

    min_dists = []
    for _, row in df_scored.iterrows():
        dists = haversine_vectorized(row['lat'], row['lot'], df_geocoded['lat_c'], df_geocoded['lon_c'])
        min_dists.append(np.min(dists))

    df_scored['dist_to_real'] = min_dists
    
    # 20m 기준 검증
    threshold = 20
    df_scored['is_real_exists'] = df_scored['dist_to_real'] <= threshold
    df_scored['model_predicted'] = df_scored['ev_충전기_설치수'] > 0

    # 지표 계산
    tp = len(df_scored[(df_scored['model_predicted'] == True) & (df_scored['is_real_exists'] == True)])
    fp = len(df_scored[(df_scored['model_predicted'] == True) & (df_scored['is_real_exists'] == False)])
    fn = len(df_scored[(df_scored['is_real_exists'] == True) & (df_scored['model_predicted'] == False)])
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    
    print(f"\n[최종 검증 결과]")
    print(f"- 일치(TP): {tp}건")
    print(f"- 과잉(FP): {fp}건")
    print(f"- 누락(FN): {fn}건")
    print(f"- 정밀도(Precision): {precision:.2f}")

    df_scored.to_csv(final_val_path, index=False, encoding='utf-8-sig')
    print(f"\n전체 분석 완료! 결과 파일: {final_val_path}")

except Exception as e:
    print(f"\n오류 발생: {e}")

Step 1: 일렉베리 데이터 지오코딩 시작...
  - 20개 변환 완료...
  - 40개 변환 완료...
  - 60개 변환 완료...
  - 80개 변환 완료...
  - 100개 변환 완료...
  - 120개 변환 완료...
  - 140개 변환 완료...
-> 지오코딩 완료 및 저장: C:\Users\seonu\Documents\axis3-self-consumption\canopy\output\results\seoul_parking_geocoded.csv

Step 2: 좌표 기반 거리 매칭 검증 시작...

[최종 검증 결과]
- 일치(TP): 1건
- 과잉(FP): 115건
- 누락(FN): 0건
- 정밀도(Precision): 0.01

전체 분석 완료! 결과 파일: C:\Users\seonu\Documents\axis3-self-consumption\canopy\output\results\final_spatial_validation.csv


- 서울_EV충전소.csv (공공데이터): 환경부, 한전 등 모든 공공/민간 충전기를 포괄하는 '전수 데이터'에 가깝습니다.

- 일렉베리 (Crawling): 주로 일반 사용자가 쉽게 접근 가능한 '개방형/지역 거점' 중심의 데이터일 확률이 높습니다. 아파트 지하주차장이나 비공용 충전기는 리스트에서 빠졌을 가능성이 큽니다.

>"본 연구에서 구축한 50m 프록시 모델과 실측 데이터(일렉베리) 간의 낮은 정합성(0.03)은, 실측 데이터의 표본수(144건)가 서울시 전체 충전소 대비 극히 일부에 불과하여 발생하는 통계적 불일치로 해석된다."

>"좌표 기반 50m 반경 설정은 도심지의 높은 건물 밀도로 인해 **과잉 추정(False Positive)**의 위험이 높음을 확인하였다. 이는 단순 거리 기반 분석보다 '장소 속성 정보'가 결합된 정밀한 필터링이 필요함을 시사한다."

일렉베리 커버리지 개낮다!! 폐기폐기

---

### 3.데이터 통합 및 최종 검증 전략

3가지 파편화된 실측 데이터(일렉베리, 서울시 리서치, 시설공단)를 하나로 모아, 741개 주차장에 대한 '통합 검증 데이터셋'을 구축

목적: 단순히 "맞다/틀리다"를 넘어, 왜 틀렸는지(구조적 한계)를 구글 어스로 증명하는 것

- 통합 검증 데이터셋 구성 (Ground Truth)
    다음 순서로 실제값 컬럼을 업데이트합니다:

    - 일렉베리 (144개): 좌표 매칭을 통한 50m이내 존재 여부 판별

    - 서울시 리서치 (9개): 명칭 기반 강제 업데이트 ('동작갯마을', '수서역' 등)

    - 서울시설공단 (25개): 과거 매칭 리스트를 기반으로 업데이트


>"단일 소스(일렉베리) 사용 시보다 서울시 리서치 및 시설공단 데이터를 통합했을 때 정밀도가 향상되었으나, 대규모 부지 및 고밀도 역세권에서의 프록시 오차는 여전히 공간적 특성(부지 크기, 인접 빌딩)에 기인함을 구글 어스 시각화를 통해 증명함."

#### 3.1 데이터 통합- 실제값 칼럼 생성

In [6]:
import pandas as pd
import numpy as np

# 하버사인 거리 계산 함수
def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371000 
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

# 경로 설정
scored_path = r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\data\processed\parking_axis3_scored.csv'
geocoded_path = r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\output\results\seoul_parking_geocoded.csv'
output_path = r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\output\results\final_integrated_validation.csv'

# [리서치 리스트] 서울시 리서치 9개소
research_names = [
    '동작갯마을 공영주차장', '홍은2동제3공영주차장', '서울은평 수색동 공영주차장',
    '문정근린공원 공영주차장', '수서역 공영주차장 복합충전소', '신천유수지공영주차장',
    '논현로22길 공영주차장', '영희초교 공영주차장', '대왕초등학교 공영주차장'
]

# [시설공단 리스트] 매칭 성공한 25개소 명칭 키워드
sisul_names = [
    '개화산역', '개화역', '구파발역', '당산노외', '도봉산역', '동대문', '볕우물', '복정역', 
    '사당노외', '세종로', '수서역북', '신대방역', '신방화역', '신천유수지', '영남', 
    '영등포구청역', '용산주차빌딩', '잠실역', '종묘', '창동역(서)', '천왕역', '천호역', 
    '학여울', '한강진역', '훈련원공원'
]

try:
    # 데이터 로드
    df = pd.read_csv(scored_path)
    df_geo = pd.read_csv(geocoded_path).dropna(subset=['lat_c', 'lon_c'])

    print(f"로드 완료: 분석 대상 {len(df)}건")

    # Step 1: 일렉베리 좌표 기반 매칭 (50m)
    print("Step 1: 일렉베리 좌표 매칭 중...")
    def check_geocoded(row):
        dists = haversine_vectorized(row['lat'], row['lot'], df_geo['lat_c'], df_geo['lon_c'])
        return 'Y' if np.min(dists) <= 50 else 'N'
    
    df['실제값'] = df.apply(check_geocoded, axis=1)

    # Step 2: 리서치 및 시설공단 리스트 통합 업데이트
    print("Step 2: 리서치 및 시설공단 데이터 통합 중...")
    combined_research_list = research_names + sisul_names
    
    def update_real_value(row):
        # 이미 좌표 매칭으로 Y라면 통과
        if row['실제값'] == 'Y':
            return 'Y'
        
        # 주차장 명칭에 리스트의 키워드가 포함되어 있는지 확인
        for target in combined_research_list:
            if target.replace(" ", "") in row['pklt_nm'].replace(" ", ""):
                return 'Y'
        return 'N'

    df['실제값'] = df.apply(update_real_value, axis=1)

    # Step 3: 최종 파일 저장
    df.to_csv(output_path, index=False, encoding='utf-8-sig')
    
    y_count = len(df[df['실제값'] == 'Y'])
    print("-" * 30)
    print(f" 통합 완료: {output_path}")
    print(f" 최종 실제 설치 확인(Y): {y_count}건")

except Exception as e:
    print(f" 오류 발생: {e}")

로드 완료: 분석 대상 741건
Step 1: 일렉베리 좌표 매칭 중...
Step 2: 리서치 및 시설공단 데이터 통합 중...
------------------------------
 통합 완료: C:\Users\seonu\Documents\axis3-self-consumption\canopy\output\results\final_integrated_validation.csv
 최종 실제 설치 확인(Y): 30건


### 3.2 구글 어스 최종 시각화: 통합 실측 데이터 기반 분석

이 단계에서는 통합된 30개의 실측값('Y')과 모델의 추정값(ev_50m > 0)을 비교하여, 공간적 오차의 원인을 규명합니다.

1. 4색 핀 분류 체계 (최종)
    - Case A (Green): 모델 예측 성공 (프록시 > 0, 실제 Y) $\rightarrow$ 신뢰성 증거
    - Case B (Red): 과잉 추정 (프록시 > 0, 실제 N) $\rightarrow$ 역세권 빌딩 간섭
    - Case C (Orange): 누락 (프록시 = 0, 실제 Y) $\rightarrow$ 대형 부지 오차
    - Case D (Gray): 정상 제외 (프록시 = 0, 실제 N) $\rightarrow$ 대다수의 일반 주차장

In [9]:
import pandas as pd

# 통합 검증 데이터 로드
final_val_path = r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\output\results\final_integrated_validation.csv'
df = pd.read_csv(final_val_path)

def create_final_kml(df, output_path):
    # KML 스타일 및 헤더 정의
    kml_content = """<?xml version="1.0" encoding="UTF-8"?>
<kml xmlns="http://www.opengis.net/kml/2.2">
<Document>
    <name>Final Integrated Validation</name>
    <Style id="color_A"><IconStyle><color>ff00ff00</color><scale>1.2</scale><Icon><href>http://maps.google.com/mapfiles/kml/paddle/wht-circle.png</href></Icon></IconStyle></Style>
    <Style id="color_B"><IconStyle><color>ff0000ff</color><scale>1.2</scale><Icon><href>http://maps.google.com/mapfiles/kml/paddle/wht-circle.png</href></Icon></IconStyle></Style>
    <Style id="color_C"><IconStyle><color>ff00a5ff</color><scale>1.2</scale><Icon><href>http://maps.google.com/mapfiles/kml/paddle/wht-circle.png</href></Icon></IconStyle></Style>
    <Style id="color_D"><IconStyle><color>ff888888</color><scale>0.8</scale><Icon><href>http://maps.google.com/mapfiles/kml/paddle/wht-circle.png</href></Icon></IconStyle></Style>
"""
    for _, row in df.iterrows():
        proxy_val = row['ev_충전기_설치수']
        real_val = str(row['실제값']).upper()
        
        # 조건별 분류
        if proxy_val > 0 and real_val == 'Y':
            case_label, style = "Case A: 성공 (Match)", "#color_A"
        elif proxy_val > 0 and real_val == 'N':
            case_label, style = "Case B: 과잉 (Over-estimated)", "#color_B"
        elif proxy_val == 0 and real_val == 'Y':
            case_label, style = "Case C: 누락 (Missed)", "#color_C"
        else:
            case_label, style = "Case D: 제외 (Correct Negative)", "#color_D"
        
        # [수정 포인트] 따옴표 3개를 사용하여 SyntaxError 방지
        desc = f"""<![CDATA[
            <h3>{row['pklt_nm']}</h3>
            <p><b>분석 결과:</b> {case_label}</p>
            <p><b>50m 프록시 설치수:</b> {proxy_val}대</p>
            <p><b>통합 실측 여부:</b> {real_val}</p>
            <p><b>좌표:</b> {row['lat']}, {row['lot']}</p>
        ]]>"""
        
        kml_content += f"""
    <Placemark>
        <name>{row['pklt_nm']}</name>
        <description>{desc}</description>
        <styleUrl>{style}</styleUrl>
        <Point><coordinates>{row['lot']},{row['lat']},0</coordinates></Point>
    </Placemark>"""
    
    kml_content += "\n</Document>\n</kml>"
    
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(kml_content)

# 실행
output_kml = r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\output\results\final_integrated_viz.kml'
create_final_kml(df, output_kml)
print(f" 구글 어스 시각화 파일 생성 완료: {output_kml}")

 구글 어스 시각화 파일 생성 완료: C:\Users\seonu\Documents\axis3-self-consumption\canopy\output\results\final_integrated_viz.kml


"통합 실측 데이터(30개소)를 바탕으로 분석한 결과, 50m 프록시 모델은 도심지의 공간적 밀집도(Spatial Density)와 필지 규모(Parcel Size)에 따라 오차가 발생하는 것으로 나타났다. 특히 역세권 고밀 지역에서의 과잉 추정과 대형 부지에서의 누락은 단순 반경 분석의 한계를 보여주나, 전반적인 충전소 분포 경향을 파악하는 데는 유의미한 프록시 변수임을 시각적으로 확인하였다."